In [ ]:
import os
import random
import time

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score
)
import tensorflow as tf
from tensorflow.keras import models, Sequential, applications, layers
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    CSVLogger,
    ReduceLROnPlateau
)
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.utils import image_dataset_from_directory
import keras_tuner as kt

In [ ]:
TRAIN_DATA_PATH = "../affectnet_dataset/Train"
TEST_DATA_PATH = "../affectnet_dataset/Test"
EPOCHS = 100
RANDOM_SEED = 40
BATCH_SIZE = 32
IMG_SIZE = (96,96)

SAVED_MODEL = "../saved_models/transfer_kt_model.keras"


In [ ]:
# Set global random seeds for reproducibility
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)

In [ ]:
'''
flow_from_directory is older approach - slower
used image_dataset_from_directory instead for better performance and easier handling of validation split
'''

train_dataset = image_dataset_from_directory(
    TRAIN_DATA_PATH,
    validation_split=0.2,
    subset="training",
    seed = RANDOM_SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode = "int",
    shuffle=True
)

val_dataset = image_dataset_from_directory(
    TRAIN_DATA_PATH,
    validation_split=0.2,
    seed = RANDOM_SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode = "int",
    subset="validation",
    shuffle=False
)

test_dataset = image_dataset_from_directory(
    TEST_DATA_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode = "int"
)

In [ ]:
train_counts = {}
CLASSES = train_dataset.class_names

for cls in CLASSES:
    cls_folder = os.path.join(TRAIN_DATA_PATH, cls)
    train_counts[cls] = len(os.listdir(cls_folder))

print(train_counts)
num_classes = len(CLASSES)

In [ ]:
print(train_dataset.class_names)

In [ ]:
# '''
# Compute class weights from train splitted part only (imbalanced data)
# - concetenate() is used to combine all batches of labels into array
# for class weight computation
# - y.numpy() is used to convert tensor to numpy array for class weight computation
# Class weights змушують loss сильніше штрафувати помилки на rare classes.
# '''

# train_labels = np.concatenate([y.numpy() for _, y in train_dataset], axis=0)

# class_weights_array = compute_class_weight(
#     class_weight="balanced",
#     classes=np.arange(num_classes),
#     y=train_labels
# )
# class_weights = {i: w for i, w in enumerate(class_weights_array)}

# print("Class names:", CLASSES)
# print("Class weights:", class_weights)

In [ ]:
plt.bar(train_counts.keys(), train_counts.values(), color='skyblue')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.title('Distribution of Training Samples')
plt.show()

In [ ]:
data_augmentation = Sequential([
    layers.RandomFlip("horizontal", seed=RANDOM_SEED),
    layers.RandomRotation(0.05, seed=RANDOM_SEED), # 0.1
    layers.RandomZoom(0.05, seed=RANDOM_SEED),
], name="data_augmentation")

In [ ]:
base_model = applications.MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False, # rm last classification block
    weights = "imagenet" # take weights pretrained on imagenet
)
# Freeze base block
base_model.trainable = False

In [ ]:
# Functional API for more flexibility in architecture and hp tuning
def build_model(hp):
    inputs = layers.Input(shape=(*IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = preprocess_input(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        128,
        kernel_regularizer=tf.keras.regularizers.l2(
            hp.Float('l2_1', 1e-5, 1e-4, sampling='log')
        )
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(hp.Float('dropout1', 0.1, 0.4, step=0.1))(x)

    x = layers.Dense(
        64,
        kernel_regularizer=tf.keras.regularizers.l2(
            hp.Float('l2_2', 1e-5, 1e-4, sampling='log')
        )
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(hp.Float('dropout2', 0.1, 0.4, step=0.1))(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=AdamW(
            learning_rate=hp.Float('lr1', 1e-5, 1e-3, sampling='log')
        ),
        metrics=['accuracy']
    )
    return model

In [ ]:
'''
Initialize a tuner (here, HyperBand). 
We use objective to specify the objective to select the best models,
and we use max_trials to specify the number of different models to try.
'''
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=20,
    factor=3, # Reduction factor 3x fewer models in the next round
    directory='keras_tuner_results',
    project_name='emotion_recognition',
    overwrite=True
)

In [ ]:
tuner.search_space_summary()

In [ ]:
tuner.search(
    train_dataset,
    epochs=20,
    validation_data=val_dataset
)

In [ ]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = tuner.hypermodel.build(best_hp)
best_model.summary()

In [ ]:
best_hp.values

In [ ]:
tuner.results_summary()

In [ ]:
callbacks_stage1 = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        SAVED_MODEL, 
        monitor='val_loss', 
        save_best_only=True
    ),
    CSVLogger('training_log_stage1.csv'),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3, #0.5
        patience=2, #5
        min_lr=1e-6
)
]

In [ ]:
# STAGE 1 - training frozen backbone
start = time.time()
history_stage1 = best_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    callbacks=callbacks_stage1
)
end = time.time()
elapsed_time = end - start

print("Stage 1 Training time: ", time.strftime("%H:%M:%S", time.gmtime(elapsed_time)))

In [ ]:
plt.figure(figsize=(12,5))

# Loss
plt.subplot(1,2,1)
plt.plot(history_stage1.history['loss'], label='train_loss')
plt.plot(history_stage1.history['val_loss'], label='val_loss')
plt.title('Stage 1 Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Accuracy
plt.subplot(1,2,2)
plt.plot(history_stage1.history['accuracy'], label='train_acc')
plt.plot(history_stage1.history['val_accuracy'], label='val_acc')
plt.title('Stage 1 Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

In [ ]:
# STAGE 2 - Last 30 layers of base model fine-tuned
base_model.trainable = True

# Freeze all layers except last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Freeze batch normalization layers, to stabilize training and prevent overfitting when fine-tuning with small datasets
for layer in base_model.layers[-30:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

best_model.compile(
    optimizer=AdamW(
        learning_rate=1e-5,
        weight_decay=1e-5
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
callbacks_stage2 = [
    EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        SAVED_MODEL,
        monitor="val_loss",
        save_best_only=True
    ),
    CSVLogger("training_log_stage2.csv"),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

In [ ]:
start = time.time()
history_stage2 = best_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    class_weight=class_weights,
    callbacks=callbacks_stage2
)
end = time.time()
print("Stage 2 training time:", time.strftime("%H:%M:%S", time.gmtime(end - start)))

In [ ]:
plt.figure(figsize=(12,5))

# Loss
plt.subplot(1,2,1)
plt.plot(history_stage2.history['loss'], label='train_loss')
plt.plot(history_stage2.history['val_loss'], label='val_loss')
plt.title('Stage 2 Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Accuracy
plt.subplot(1,2,2)
plt.plot(history_stage2.history['accuracy'], label='train_acc')
plt.plot(history_stage2.history['val_accuracy'], label='val_acc')
plt.title('Stage 2 Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

In [ ]:
model = models.load_model(SAVED_MODEL)

In [ ]:
test_loss, test_acc = best_model.evaluate(test_dataset)
print(f"Test Accuracy: {test_acc*100:.2f}%")

In [ ]:
def evaluate_model(model, dataset, class_names):
    print("Evaluating on test set...")
    test_loss, test_acc = model.evaluate(dataset, verbose=1)
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc * 100:.2f}%")

    print("\nGenerating predictions...")
    y_prob = model.predict(dataset, verbose=1)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = np.concatenate([y.numpy() for _, y in dataset], axis=0)

    macro_f1 = f1_score(y_true, y_pred, average="macro")
    print(f"\nMacro F1: {macro_f1:.4f}")

    print("\nClassification Report:\n")
    print(classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        zero_division=0
    ))

    cm_norm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(len(class_names)),
        normalize="true"
    )

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Normalized Confusion Matrix")
    plt.tight_layout()
    plt.show()

    return test_loss, test_acc, macro_f1, cm_norm

test_loss, test_acc, macro_f1, cm_norm = evaluate_model(model, test_dataset, CLASSES)

In [ ]:
def predict_random_samples():

    # Get one batch
    images, labels = next(iter(test_dataset))

    # Random 5 images
    np.random.seed(RANDOM_SEED)
    indices = np.random.choice(len(images), 5, replace=False)

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))

    for i, idx in enumerate(indices):

        img = images[idx]

        # True label
        true_idx = labels[idx].numpy()
        true_label = CLASSES[true_idx]

        # Prediction
        pred_prob = model.predict(
            np.expand_dims(img, axis=0),
            verbose=0
        )

        pred_idx = np.argmax(pred_prob)
        pred_label = CLASSES[pred_idx]

        # Plot image
        axes[i].imshow(img.numpy().astype("uint8"))
        axes[i].axis("off")

        # Color
        color = "green" if true_label == pred_label else "red"

        axes[i].set_title(
            f"True: {true_label}\nPred: {pred_label}",
            color=color
        )

    plt.tight_layout()
    plt.show()

predict_random_samples()